# Modelo final — TCC II (DENTEX + YOLOv8s, 1.024 px)

Este notebook fecha o modelo final em uma única execução (**Save & Run All**):

1. **Divisão do teste interno (plano B):** separa 15% das 705 imagens de treino como **teste**, de forma estratificada (seed 42). O teste oficial do DENTEX não é usado, porque o mapeamento dos rótulos em turco não está documentado.
2. **Treino do modelo final:** a configuração escolhida nos experimentos de configuração (exp. 5: YOLOv8s, 1.024 px, 100 épocas). O `best.pt` é escolhido na validação oficial (50 imagens), como antes.
3. **Limiar de confiança:** escolhido **na validação**, como o que maximiza a média do F1 entre as quatro classes.
4. **Pacote do especialista:** 25 imagens do teste interno, com as caixas numeradas e **sem as anotações**, um PDF para impressão, uma planilha para preenchimento e, separado, o arquivo de cruzamento com as anotações (só para o autor).
5. **Avaliação no teste:** roda com `AVALIAR_TESTE = True` (Célula 3); o plano B foi adotado em 09/11/2026. Como o treino é determinístico, rodar de novo produz o mesmo modelo. Além das métricas e da matriz de confusão, salva as predições e as anotações do teste (`teste_predicoes.json`) e cópias reduzidas das 106 imagens (`imagens_teste/`), usadas nas figuras de acertos e erros e na análise por região da arcada, sem precisar de GPU.

**Como rodar:** Settings → Accelerator **GPU T4 x2**, Internet **On** → **Save Version → Save & Run All**. Leva cerca de 1 a 1,5 h. Baixe da aba **Output**: `pacote_especialista.zip` (vai para o dentista) e `artefatos_modelo_final.zip` (fica com você).

Para não retreinar, suba o `best.pt` como dataset do Kaggle e informe o caminho em `PESOS_PRONTOS` (Célula 3). **Atenção:** só use pesos treinados com a MESMA divisão (este notebook); o `best.pt` do experimento 5 viu as imagens do teste interno no treino e não serve aqui.


In [ ]:
!pip install -q ultralytics==8.4.183 remotezip huggingface_hub pyyaml pycocotools openpyxl


In [ ]:
# !pip install -q remotezip huggingface_hub pyyaml   (rode se faltar algum)
import json, shutil, time
from collections import Counter
from pathlib import Path, PurePosixPath

import yaml
from huggingface_hub import hf_hub_download, hf_hub_url
from remotezip import RemoteZip

REPO = "ibrahimhamamci/DENTEX"
OUT = Path("/kaggle/working/dentex_yolo")

# ---- Decisão metodológica em UM lugar só --------------------------------
# chave = category_id_3 original do DENTEX; valor = id da classe no YOLO (None = descartar)
CLASS_MAP = {0: 0, 1: 1, 2: 2, 3: 3}
YOLO_NAMES = {0: "impacted", 1: "caries", 2: "periapical_lesion", 3: "deep_caries"}
# Alternativas (basta trocar as duas linhas acima, sem rebaixar nada):
#  (b) sem impactado:       CLASS_MAP = {0: None, 1: 0, 2: 1, 3: 2}
#                           YOLO_NAMES = {0: "caries", 1: "periapical_lesion", 2: "deep_caries"}
#  (c) cáries unificadas:   CLASS_MAP = {0: 0, 1: 1, 2: 2, 3: 1}
#                           YOLO_NAMES = {0: "impacted", 1: "caries", 2: "periapical_lesion"}

# Pastas dentro dos ZIPs (atenção: o ZIP de treino tem 3 subconjuntos com nomes de arquivo parecidos)
TRAIN_ZIP, TRAIN_IMG_DIR = "training_data.zip", "training_data/quadrant-enumeration-disease/xrays/"
TRAIN_JSON = "training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json"
VAL_ZIP, VAL_IMG_DIR = "validation_data.zip", "validation_data/quadrant_enumeration_disease/xrays/"


def coco_para_yolo(bbox, w_img, h_img):
    """COCO [x_min, y_min, largura, altura] em pixels -> YOLO [cx, cy, w, h] normalizado em [0, 1]."""
    x, y, w, h = bbox
    x0, y0 = max(0.0, x), max(0.0, y)                      # recorta caixas que escapam da imagem
    x1, y1 = min(float(w_img), x + w), min(float(h_img), y + h)
    if x1 <= x0 or y1 <= y0:
        return None
    return ((x0 + x1) / 2 / w_img, (y0 + y1) / 2 / h_img, (x1 - x0) / w_img, (y1 - y0) / h_img)


def converter(coco, zip_name, img_dir, split):
    """Extrai as imagens do split e grava um .txt de rótulos por imagem."""
    (OUT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT / "labels" / split).mkdir(parents=True, exist_ok=True)

    por_imagem = {}
    for a in coco["annotations"]:
        por_imagem.setdefault(a["image_id"], []).append(a)

    url = hf_hub_url(REPO, f"DENTEX/{zip_name}", repo_type="dataset")
    stats = Counter()
    with RemoteZip(url) as z:
        membros = {PurePosixPath(n).name: n for n in z.namelist() if n.startswith(img_dir) and n.endswith(".png")}
        for img in coco["images"]:
            nome = img["file_name"]
            destino = OUT / "images" / split / nome
            if not destino.exists():                        # idempotente: reexecutar não rebaixa
                for tentativa in range(3):
                    try:
                        with z.open(membros[nome]) as src, open(destino, "wb") as dst:
                            shutil.copyfileobj(src, dst)
                        break
                    except Exception as e:                  # rede instável: tenta de novo
                        print("retry", nome, e); time.sleep(2)
            linhas = []
            for a in por_imagem.get(img["id"], []):
                cls = CLASS_MAP[a["category_id_3"]]
                if cls is None:
                    stats["descartadas"] += 1
                    continue
                yolo = coco_para_yolo(a["bbox"], img["width"], img["height"])
                if yolo is None:
                    stats["caixas_invalidas"] += 1
                    continue
                linhas.append(f"{cls} " + " ".join(f"{v:.6f}" for v in yolo))
                stats[f"classe_{cls}"] += 1
            (OUT / "labels" / split / (Path(nome).stem + ".txt")).write_text("\n".join(linhas))  # vazio = fundo
            stats["imagens"] += 1
            stats["imagens_sem_caixa"] += (len(linhas) == 0)
    return stats


# ---- Treino ----
with RemoteZip(hf_hub_url(REPO, f"DENTEX/{TRAIN_ZIP}", repo_type="dataset")) as z:
    train_coco = json.loads(z.read(TRAIN_JSON))
val_coco = json.load(open(hf_hub_download(REPO, "DENTEX/validation_triple.json", repo_type="dataset")))

st_train = converter(train_coco, TRAIN_ZIP, TRAIN_IMG_DIR, "train")
st_val = converter(val_coco, VAL_ZIP, VAL_IMG_DIR, "val")
print("treino:", dict(st_train)); print("validação:", dict(st_val))

# ---- data.yaml: é ele que o YOLO lê no treino ----
(OUT / "data.yaml").write_text(yaml.safe_dump(
    {"path": str(OUT), "train": "images/train", "val": "images/val", "names": YOLO_NAMES},
    sort_keys=False, allow_unicode=True))
print((OUT / "data.yaml").read_text())
# Guarda o gabarito da validação em COCO: a avaliação com pycocotools (Célula 4) usa este arquivo.
(OUT / "val_coco.json").write_text(json.dumps(val_coco))
(OUT / "train_coco.json").write_text(json.dumps(train_coco))   # gabarito do teste interno (Célula 4)


In [ ]:
import json, random, re, shutil, time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from sklearn.model_selection import train_test_split

SEED = 42
FRACAO_TESTE = 0.15                 # ~106 das 705 imagens de treino viram teste interno
MODELO, IMGSZ, EPOCAS, BATCH = "yolov8s.pt", 1024, 100, 16   # configuração do exp. 5
DEVICE = 0
PESOS_PRONTOS = None                # ex.: "/kaggle/input/meu-dataset/best.pt" (só se treinado com esta divisão)
AVALIAR_TESTE = True                # plano B (adotado em 09/11/2026): avalia o teste interno uma única vez
N_ESPECIALISTA = 25                 # imagens do teste interno para o cirurgião-dentista

RUNS = Path("/kaggle/working/runs")
ART = Path("/kaggle/working/artefatos_modelo_final"); ART.mkdir(parents=True, exist_ok=True)
PAC = Path("/kaggle/working/pacote_especialista"); PAC.mkdir(parents=True, exist_ok=True)
NOMES_PT = {0: "dente impactado", 1: "cárie", 2: "lesão periapical", 3: "cárie profunda"}
random.seed(SEED); np.random.seed(SEED)


In [ ]:
# ---- Divisão estratificada: cada imagem recebe o "estrato" da classe mais rara que contém ----
PRIORIDADE = [2, 3, 0, 1]           # periapical, cárie profunda, impactado, cárie (da mais rara à mais comum)

def classes_da_imagem(txt):
    return {int(l.split()[0]) for l in txt.read_text().splitlines() if l.strip()}

def estrato(cls):
    for c in PRIORIDADE:
        if c in cls:
            return str(c)
    return "fundo"

imgs = sorted((OUT / "images" / "train").glob("*.png"))
cls_por_img = {p.stem: classes_da_imagem(OUT / "labels" / "train" / f"{p.stem}.txt") for p in imgs}
nomes = [p.stem for p in imgs]
treino, teste = train_test_split(nomes, test_size=FRACAO_TESTE, random_state=SEED,
                                 stratify=[estrato(cls_por_img[n]) for n in nomes])

for sub in ["images", "labels"]:
    (OUT / sub / "test").mkdir(parents=True, exist_ok=True)
for n in teste:                      # move (não copia): o teste sai do treino
    shutil.move(OUT / "images" / "train" / f"{n}.png", OUT / "images" / "test" / f"{n}.png")
    shutil.move(OUT / "labels" / "train" / f"{n}.txt", OUT / "labels" / "test" / f"{n}.txt")

(OUT / "data.yaml").write_text(yaml.safe_dump(
    {"path": str(OUT), "train": "images/train", "val": "images/val", "test": "images/test", "names": YOLO_NAMES},
    sort_keys=False, allow_unicode=True))

def resumo_split(split):
    caixas, imagens = Counter(), Counter()
    arqs = list((OUT / "labels" / split).glob("*.txt"))
    for f in arqs:
        linhas = [int(l.split()[0]) for l in f.read_text().splitlines() if l.strip()]
        caixas.update(linhas); imagens.update(set(linhas))
    return {"imagens": len(arqs), **{f"caixas_{NOMES_PT[c]}": caixas[c] for c in range(4)},
            **{f"imgs_{NOMES_PT[c]}": imagens[c] for c in range(4)}}

divisao = pd.DataFrame({s: resumo_split(s) for s in ["train", "val", "test"]}).T
print(divisao.to_string())
divisao.to_csv(ART / "divisao_resumo.csv")
pd.DataFrame({"imagem": teste}).to_csv(ART / "teste_interno_imagens.csv", index=False)
assert not set(teste) & {p.stem for p in (OUT / "images" / "train").glob("*.png")}, "vazamento treino/teste"


In [ ]:
from ultralytics import YOLO

if PESOS_PRONTOS:
    PESOS = PESOS_PRONTOS
    print("Usando pesos prontos:", PESOS)
else:
    YOLO(MODELO).train(data=str(OUT / "data.yaml"), imgsz=IMGSZ, epochs=EPOCAS, patience=EPOCAS,
                       batch=BATCH, seed=SEED, deterministic=True, device=DEVICE, workers=2,
                       project=str(RUNS), name="final", exist_ok=True)
    PESOS = str(RUNS / "final" / "weights" / "best.pt")
    for n in ["results.csv", "results.png", "args.yaml", "weights/best.pt"]:
        if (RUNS / "final" / n).exists():
            shutil.copy(RUNS / "final" / n, ART / Path(n).name)
    df = pd.read_csv(RUNS / "final" / "results.csv"); df.columns = [c.strip() for c in df.columns]
    print("melhor época:", int(df["metrics/mAP50-95(B)"].idxmax()) + 1)
modelo = YOLO(PESOS)


In [ ]:
# ---- Predições e pareamento com o gabarito (usados no limiar, no teste e no cruzamento) ----
def gabarito(coco, nomes_img):
    """Caixas anotadas em pixels da imagem original, por nome de arquivo, com a numeração FDI do dente."""
    ids = {im["id"]: im["file_name"] for im in coco["images"] if Path(im["file_name"]).stem in nomes_img}
    num = lambda nome, i: (re.sub(r"\D", "", str(nome)) or str(i + 1))
    q = {c["id"]: num(c["name"], c["id"]) for c in coco.get("categories_1", [])}
    d = {c["id"]: num(c["name"], c["id"]) for c in coco.get("categories_2", [])}
    gt = {Path(f).stem: [] for f in ids.values()}
    for a in coco["annotations"]:
        if a["image_id"] in ids and CLASS_MAP[a["category_id_3"]] is not None:
            x, y, w, h = a["bbox"]
            fdi = f'{q.get(a.get("category_id_1"), "?")}{d.get(a.get("category_id_2"), "?")}'
            gt[Path(ids[a["image_id"]]).stem].append(
                {"cls": CLASS_MAP[a["category_id_3"]], "xyxy": [x, y, x + w, y + h], "fdi": fdi})
    return gt

def predizer(pasta, nomes_img):
    out = {}
    fontes = [str(pasta / f"{n}.png") for n in nomes_img]
    for r in modelo.predict(source=fontes, imgsz=IMGSZ, conf=0.001, iou=0.7, max_det=300,
                            device=DEVICE, stream=True, verbose=False):
        out[Path(r.path).stem] = [{"cls": int(c), "conf": float(s), "xyxy": b}
                                  for b, c, s in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist())]
    return out

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter + 1e-9)

def parear(preds, gt, limiar_iou=0.5):
    """Marca cada predição como VP ou FP (mesma classe, IoU >= 0,5, gulosa por confiança).
    A marcação não depende do limiar de confiança, então basta fazê-la uma vez."""
    linhas, n_gt = [], Counter()
    for img in gt:
        usados = set()
        for g in gt[img]:
            n_gt[g["cls"]] += 1
        for p in sorted(preds.get(img, []), key=lambda p: -p["conf"]):
            melhor, j_melhor = 0.0, None
            for j, g in enumerate(gt[img]):
                if j not in usados and g["cls"] == p["cls"]:
                    v = iou(p["xyxy"], g["xyxy"])
                    if v > melhor:
                        melhor, j_melhor = v, j
            vp = melhor >= limiar_iou
            if vp:
                usados.add(j_melhor)
            linhas.append({"imagem": img, "cls": p["cls"], "conf": p["conf"], "vp": vp})
    return pd.DataFrame(linhas), n_gt

def metricas_no_limiar(pareado, n_gt, t):
    res = {}
    for c in range(4):
        sel = pareado[(pareado.cls == c) & (pareado.conf >= t)]
        vp = int(sel.vp.sum()); fp = len(sel) - vp; fn = n_gt[c] - vp
        p = vp / (vp + fp) if vp + fp else 0.0; r = vp / (vp + fn) if vp + fn else 0.0
        res[c] = {"VP": vp, "FP": fp, "FN": fn, "precisao": p, "revocacao": r,
                  "F1": 2 * p * r / (p + r) if p + r else 0.0}
    return res


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

val_coco = json.loads((OUT / "val_coco.json").read_text())
nomes_val = [p.stem for p in (OUT / "images" / "val").glob("*.png")]
gt_val = gabarito(val_coco, set(nomes_val))
pred_val = predizer(OUT / "images" / "val", nomes_val)
par_val, ngt_val = parear(pred_val, gt_val)

limiares = np.round(np.arange(0.05, 0.951, 0.01), 2)
curva = []
for t in limiares:
    m = metricas_no_limiar(par_val, ngt_val, t)
    curva.append({"limiar": t, "F1_medio": np.mean([m[c]["F1"] for c in range(4)]),
                  **{f"F1_{NOMES_PT[c]}": m[c]["F1"] for c in range(4)}})
curva = pd.DataFrame(curva)
LIMIAR = float(curva.loc[curva.F1_medio.idxmax(), "limiar"])
curva.to_csv(ART / "curva_limiar_validacao.csv", index=False)
print(f"Limiar escolhido na validação: {LIMIAR:.2f} (F1 médio {curva.F1_medio.max():.3f})")
m = metricas_no_limiar(par_val, ngt_val, LIMIAR)
tab_val = pd.DataFrame({NOMES_PT[c]: m[c] for c in range(4)}).T
print(tab_val.round(3).to_string()); tab_val.to_csv(ART / "validacao_no_limiar.csv")

# Figura para a monografia: F1 por classe em função do limiar
virg = FuncFormatter(lambda v, _: f"{v:.1f}".replace(".", ","))
cores = {0: "#2a78d6", 1: "#eb6834", 2: "#1baf7a", 3: "#eda100"}
fig, ax = plt.subplots(figsize=(6.3, 3.4))
for c in range(4):
    ax.plot(curva.limiar, curva[f"F1_{NOMES_PT[c]}"], color=cores[c], lw=1.6, label=NOMES_PT[c].capitalize())
ax.plot(curva.limiar, curva.F1_medio, color="#333333", lw=2.4, label="Média")
ax.axvline(LIMIAR, color="#8a8a85", lw=1, ls="--")
ax.set_xlabel("Limiar de confiança"); ax.set_ylabel("F1 na validação")
ax.xaxis.set_major_formatter(virg); ax.yaxis.set_major_formatter(virg)
ax.grid(axis="y", color="#e6e6e3", lw=0.8); ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8, ncol=2)
fig.tight_layout(); fig.savefig(ART / "f1-por-limiar.pdf"); fig.savefig(ART / "f1-por-limiar.png", dpi=150); plt.show()


In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

def avaliar_coco(coco, nomes_img, preds):
    """AP@50 por classe no protocolo do desafio (pycocotools), para qualquer subconjunto de imagens."""
    ims = [im for im in coco["images"] if Path(im["file_name"]).stem in nomes_img]
    ids = {im["id"] for im in ims}; id_por_nome = {Path(im["file_name"]).stem: im["id"] for im in ims}
    gt = {"images": [{k: im[k] for k in ("id", "file_name", "width", "height")} for im in ims],
          "categories": [{"id": i, "name": n} for i, n in YOLO_NAMES.items()], "annotations": []}
    for a in coco["annotations"]:
        if a["image_id"] in ids and CLASS_MAP[a["category_id_3"]] is not None:
            x, y, w, h = a["bbox"]
            gt["annotations"].append({"id": len(gt["annotations"]) + 1, "image_id": a["image_id"],
                                      "category_id": CLASS_MAP[a["category_id_3"]], "bbox": [x, y, w, h],
                                      "area": w * h, "iscrowd": 0})
    dets = [{"image_id": id_por_nome[n], "category_id": p["cls"], "score": p["conf"],
             "bbox": [p["xyxy"][0], p["xyxy"][1], p["xyxy"][2] - p["xyxy"][0], p["xyxy"][3] - p["xyxy"][1]]}
            for n, ps in preds.items() for p in ps]
    cg = COCO(); cg.dataset = gt; cg.createIndex()
    E = COCOeval(cg, cg.loadRes(dets), "bbox"); E.evaluate(); E.accumulate(); E.summarize()
    out = {"COCO_AP50": E.stats[1], "COCO_AP50-95": E.stats[0]}
    for k, cid in enumerate(E.params.catIds):
        p = E.eval["precision"][0, :, k, 0, 2]
        out[f"COCO_AP50_{NOMES_PT[cid]}"] = float(p[p > -1].mean()) if (p > -1).any() else float("nan")
    return out

res_val = avaliar_coco(val_coco, set(nomes_val), pred_val)
mv = modelo.val(data=str(OUT / "data.yaml"), imgsz=IMGSZ, split="val", device=DEVICE, plots=False,
                project=str(RUNS), name="val_final", exist_ok=True, verbose=False)
res_val.update({"mAP50": mv.box.map50, "mAP50-95": mv.box.map, "limiar": LIMIAR})
pd.Series(res_val).to_csv(ART / "validacao_metricas.csv")
print(pd.Series(res_val).round(3).to_string())


In [ ]:
# ---- Pacote do especialista: 25 imagens do TESTE interno, sem anotações ----
from matplotlib.backends.backend_pdf import PdfPages
from PIL import Image

def ajustar_rotulos(fig, ax, textos):
    """Desce cada rótulo até não sobrepor os já posicionados (caixas quase iguais escondiam números)."""
    fig.canvas.draw(); r = fig.canvas.get_renderer(); inv = ax.transData.inverted(); colocados = []
    for t in textos:
        for _ in range(60):
            bb = t.get_window_extent(renderer=r).expanded(1.05, 1.15)
            if not any(bb.overlaps(o) for o in colocados):
                break
            x, y = t.get_position()
            altura = abs(inv.transform((0, bb.y1))[1] - inv.transform((0, bb.y0))[1])
            t.set_position((x, y + altura))           # no imshow, y cresce para baixo
        colocados.append(t.get_window_extent(renderer=r))

train_coco = json.loads((OUT / "train_coco.json").read_text())
gt_teste = gabarito(train_coco, set(teste))

# Seleção estratificada pela classe mais rara de cada imagem (garante casos de todas as classes)
cotas = {"2": 6, "3": 6, "0": 4, "1": 7, "fundo": 2}
rng = random.Random(SEED); escolhidas = []
por_estrato = {}
for n in sorted(teste):
    por_estrato.setdefault(estrato({g["cls"] for g in gt_teste.get(n, [])}), []).append(n)
for e, k in cotas.items():
    lista = por_estrato.get(e, []); rng.shuffle(lista); escolhidas += lista[:k]
resto = [n for n in sorted(teste) if n not in escolhidas]; rng.shuffle(resto)
escolhidas += resto[:max(0, N_ESPECIALISTA - len(escolhidas))]
escolhidas = escolhidas[:N_ESPECIALISTA]; rng.shuffle(escolhidas)   # ordem aleatória: não revela o estrato

pred_esp = predizer(OUT / "images" / "test", escolhidas)
cores = {0: "#2a78d6", 1: "#eb6834", 2: "#1baf7a", 3: "#eda100"}
linhas_form, linhas_cruz, linhas_gt = [], [], []
(PAC / "imagens").mkdir(exist_ok=True)

with PdfPages(PAC / "avaliacao_especialista.pdf") as pdf:
    for i, nome in enumerate(escolhidas, 1):
        rot = f"imagem_{i:02d}"
        im = Image.open(OUT / "images" / "test" / f"{nome}.png").convert("L")
        caixas = [p for p in sorted(pred_esp.get(nome, []), key=lambda p: (p["xyxy"][0], p["xyxy"][1])) if p["conf"] >= LIMIAR]
        fig = plt.figure(figsize=(11.69, 8.27))                      # A4 deitado
        ax = fig.add_axes([0.02, 0.30, 0.96, 0.66]); ax.imshow(im, cmap="gray"); ax.axis("off")
        ax.set_title(f"{rot}  —  {len(caixas)} detecções (limiar {LIMIAR:.2f})".replace(".", ","), fontsize=11)
        textos = []
        for k, p in enumerate(caixas, 1):
            x0, y0, x1, y1 = p["xyxy"]
            ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec=cores[p["cls"]], lw=1.8))
            textos.append(ax.text(x0, y0 - 6, f"{k}", color="white", fontsize=8, weight="bold", zorder=5,
                                  bbox=dict(boxstyle="square,pad=0.15", fc=cores[p["cls"]], ec="none")))
            linhas_form.append({"imagem": rot, "caixa": k, "classe_predita": NOMES_PT[p["cls"]],
                                "confianca": round(p["conf"], 2), "avaliacao (correta / falso positivo / classe errada: qual / incerto)": "",
                                "avaliavel_na_panoramica (sim / nao, exigiria outro exame / incerto)": "", "comentario": ""})
            # cruzamento (só para o autor): anotação de maior IoU
            # prefere a anotação da MESMA classe com IoU >= 0,5 (um dente pode ter duas anotações)
            mesma = [g for g in gt_teste.get(nome, []) if g["cls"] == p["cls"] and iou(p["xyxy"], g["xyxy"]) >= 0.5]
            melhor = max(mesma or gt_teste.get(nome, []), key=lambda g: iou(p["xyxy"], g["xyxy"]), default=None)
            v = iou(p["xyxy"], melhor["xyxy"]) if melhor else 0.0
            linhas_cruz.append({"imagem": rot, "arquivo_dentex": nome, "caixa": k, "classe_predita": NOMES_PT[p["cls"]],
                                "confianca": round(p["conf"], 3),
                                "anotacao_dentex": NOMES_PT[melhor["cls"]] if melhor and v >= 0.5 else "sem anotação",
                                "dente_fdi_anotado": melhor["fdi"] if melhor and v >= 0.5 else "", "iou": round(v, 2),
                                "acerto_mesma_classe": bool(melhor and v >= 0.5 and melhor["cls"] == p["cls"])})
        ajustar_rotulos(fig, ax, textos)
        for g in gt_teste.get(nome, []):
            det = any(p["cls"] == g["cls"] and iou(p["xyxy"], g["xyxy"]) >= 0.5 for p in caixas)
            linhas_gt.append({"imagem": rot, "arquivo_dentex": nome, "dente_fdi": g["fdi"],
                              "classe_anotada": NOMES_PT[g["cls"]], "detectada_pelo_modelo": det})
        # quadro do formulário abaixo da imagem
        txt = "\n".join(f"{k:>2}. {NOMES_PT[p['cls']]:<17} conf. " + f"{p['conf']:.2f}".replace(".", ",") +
                        "   ( ) correta  ( ) falso positivo  ( ) classe errada: ______  ( ) incerto   |   avaliável na panorâmica? ( ) sim  ( ) não  ( ) incerto"
                        for k, p in enumerate(caixas, 1)) or "Nenhuma detecção acima do limiar."
        fig.text(0.03, 0.27, txt, va="top", fontsize=7 if len(caixas) > 12 else 8, family="DejaVu Sans Mono")
        fig.text(0.03, 0.03, "Achados NÃO detectados (dente FDI, patologia, avaliável na panorâmica?): "
                 "______________________________________________________________", fontsize=8)
        pdf.savefig(fig); plt.close(fig)
        # imagem em resolução original com as caixas, para ampliar na tela
        fig = plt.figure(figsize=(im.width / 150, im.height / 150), dpi=150)
        ax = fig.add_axes([0, 0, 1, 1]); ax.imshow(im, cmap="gray"); ax.axis("off")
        textos = []
        for k, p in enumerate(caixas, 1):
            x0, y0, x1, y1 = p["xyxy"]
            ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec=cores[p["cls"]], lw=2.2))
            textos.append(ax.text(x0, y0 - 8, f"{k} {NOMES_PT[p['cls']]}", color="white", fontsize=9, weight="bold", zorder=5,
                                  bbox=dict(boxstyle="square,pad=0.2", fc=cores[p["cls"]], ec="none")))
        ajustar_rotulos(fig, ax, textos)
        fig.savefig(PAC / "imagens" / f"{rot}.png", dpi=150); plt.close(fig)

instr = pd.DataFrame({"instrucoes": [
    "Cada imagem mostra as caixas do modelo, numeradas; a cor indica a classe (azul: dente impactado; laranja: cárie; verde: lesão periapical; amarelo: cárie profunda).",
    "As caixas envolvem o dente inteiro, não a lesão. Avalie se o dente marcado tem a condição indicada.",
    "Aba 'deteccoes': uma linha por caixa. Aba 'nao_detectados': achados que o modelo não marcou (dente pela numeração FDI).",
    "'Avaliável na panorâmica': marque 'não' quando a condição só poderia ser confirmada com outro exame (ex.: interproximal).",
    "As imagens são de uma base pública anonimizada (DENTEX). Não é laudo: é uma avaliação qualitativa para pesquisa."]})
vazios = pd.DataFrame([{"imagem": "", "dente_fdi": "", "patologia": "", "avaliavel_na_panoramica": "", "comentario": ""}] * 40)
with pd.ExcelWriter(PAC / "formulario_especialista.xlsx") as xw:
    instr.to_excel(xw, sheet_name="instrucoes", index=False)
    pd.DataFrame(linhas_form).to_excel(xw, sheet_name="deteccoes", index=False)
    vazios.to_excel(xw, sheet_name="nao_detectados", index=False)

# Só para o autor (NÃO enviar ao especialista): cruzamento com as anotações
pd.DataFrame(linhas_cruz).to_csv(ART / "cruzamento_deteccoes_x_anotacoes.csv", index=False)
pd.DataFrame(linhas_gt).to_csv(ART / "anotacoes_das_imagens_do_especialista.csv", index=False)
print(f"{len(escolhidas)} imagens; {len(linhas_form)} caixas; {len(linhas_gt)} anotações nessas imagens")


In [ ]:
# ---- Avaliação no teste interno: rode só depois de aprovar o plano B (AVALIAR_TESTE = True) ----
if AVALIAR_TESTE:
    pred_teste = predizer(OUT / "images" / "test", teste)
    res_teste = avaliar_coco(train_coco, set(teste), pred_teste)
    mt = modelo.val(data=str(OUT / "data.yaml"), imgsz=IMGSZ, split="test", device=DEVICE, plots=False,
                    project=str(RUNS), name="teste_final", exist_ok=True, verbose=False)
    res_teste.update({"mAP50": mt.box.map50, "mAP50-95": mt.box.map})
    pd.Series(res_teste).to_csv(ART / "teste_metricas.csv"); print(pd.Series(res_teste).round(3).to_string())
    par_t, ngt_t = parear(pred_teste, gabarito(train_coco, set(teste)))
    tab_t = pd.DataFrame({NOMES_PT[c]: v for c, v in metricas_no_limiar(par_t, ngt_t, LIMIAR).items()}).T
    tab_t.to_csv(ART / "teste_no_limiar.csv"); print(tab_t.round(3).to_string())
    modelo.val(data=str(OUT / "data.yaml"), imgsz=IMGSZ, split="test", device=DEVICE, conf=LIMIAR, plots=True,
               project=str(RUNS), name="teste_matriz", exist_ok=True, verbose=False)
    for n in ["confusion_matrix.png", "confusion_matrix_normalized.png"]:
        if (RUNS / "teste_matriz" / n).exists():
            shutil.copy(RUNS / "teste_matriz" / n, ART / f"teste_{n}")

    # ---- Dados para a análise de erros fora do Kaggle (figuras de acertos/erros e análise por dente) ----
    # Predições acima de 0,05 (o limiar 0,29 é aplicado depois) e anotações com a numeração FDI.
    # As imagens vão reduzidas (lado maior de 1.600 px, JPEG): bastam para figuras e cabem no zip.
    from PIL import Image
    gt_teste = gabarito(train_coco, set(teste))
    escalas = {}
    (ART / "imagens_teste").mkdir(exist_ok=True)
    for n in teste:
        im = Image.open(OUT / "images" / "test" / f"{n}.png").convert("L")
        escalas[n] = min(1.0, 1600 / max(im.size))
        im.resize((round(im.width * escalas[n]), round(im.height * escalas[n]))).save(
            ART / "imagens_teste" / f"{n}.jpg", quality=90)
    dados = {"limiar": LIMIAR, "classes": NOMES_PT,
             "imagens": {n: {"escala": escalas[n], "anotacoes": gt_teste.get(n, []),
                             "predicoes": [p for p in pred_teste.get(n, []) if p["conf"] >= 0.05]}
                         for n in teste}}
    (ART / "teste_predicoes.json").write_text(json.dumps(dados, ensure_ascii=False))
    print("salvos:", len(teste), "imagens e teste_predicoes.json")
else:
    print("Teste interno NÃO avaliado (AVALIAR_TESTE = False).")


In [ ]:
# Última célula: apaga a base convertida (é recriada a cada execução) e gera os dois zips
shutil.rmtree(OUT, ignore_errors=True)
shutil.make_archive("/kaggle/working/pacote_especialista", "zip", PAC)
shutil.make_archive("/kaggle/working/artefatos_modelo_final", "zip", ART)
print("pacote:", sorted(str(p.relative_to(PAC)) for p in PAC.rglob("*") if p.is_file())[:5], "...")
print("artefatos:", sorted(p.name for p in ART.iterdir()))
